### Background and Prior Work

#### Background
Motor-vehicle collisions represent a significant public health challenge in urban environments, contributing to thousands of injuries and fatalities annually. In Seattle, the Seattle Department of Transportation (SDOT) maintains comprehensive records of collision incidents, including spatial coordinates, severity metrics (e.g., fatalities and injuries), and temporal details. Understanding the spatial distribution of these collisions is crucial for urban planning, as it allows for targeted safety interventions such as improved signage, traffic calming measures, or infrastructure upgrades in high-risk areas.

In recent years, SDOT has introduced several policies to mitigate collision risks, building on broader initiatives like the Vision Zero program adopted in 2015. Vision Zero aims to eliminate traffic deaths and serious injuries by 2030 through a combination of engineering improvements, education, and enforcement. Key measures include prohibiting right turns on red lights at select high-risk intersections to protect pedestrians and cyclists, expanding protected bike lanes, and implementing pedestrian-priority designs in downtown and residential areas. These policies may have influenced collision patterns, particularly in urban cores, by altering driver behavior and reducing exposure at vulnerable points.

The COVID-19 pandemic introduced unprecedented disruptions to daily mobility patterns. Lockdowns, remote work shifts, and reduced commuting led to lower overall traffic volumes, particularly in city centers. However, anecdotal reports and preliminary studies suggest that while collision frequencies may have decreased, the severity of incidents could have shifted due to changes in driver behavior, such as increased speeding on less congested roads or altered travel patterns favoring highways over local streets. This project examines whether these changes persisted into the post-2020 period and whether spatial hotspots for collisions also correspond to disproportionate shares of severe outcomes.

#### Prior Work
Spatial analysis of traffic collisions has been extensively studied using Geographic Information Systems (GIS) and statistical methods. Research by Erdogan et al. (2008) demonstrated the use of kernel density estimation to identify collision hotspots in urban areas, revealing that a small percentage of road segments account for a large proportion of incidents. Similarly, studies in cities like Los Angeles and New York have employed spatial clustering techniques (e.g., Getis-Ord Gi* statistic) to pinpoint high-risk zones, often linked to factors such as intersection density and traffic volume (Aguero-Valverde & Jovanis, 2006).

Regarding collision severity, prior work highlights that while collision frequency is influenced by exposure (e.g., vehicle miles traveled), severity depends on factors like speed, vehicle type, and environmental conditions. For instance, analyses of the Fatality Analysis Reporting System (FARS) data show that fatalities are more likely in rural or high-speed areas, but urban settings exhibit higher injury rates due to pedestrian involvement (NHTSA, 2022). Vehicle type also plays a role, with larger vehicles (e.g., trucks) associated with higher fatality risks in multi-vehicle crashes (Blower & Green, 2010).

Post-COVID research on traffic safety is emerging. Studies in the UK and US indicate a decline in overall collisions during 2020-2021, attributed to reduced travel, but some regions reported increased severity due to riskier driving behaviors (e.g., higher speeds on empty roads) (Transport for London, 2021; NHTSA, 2021). Spatial shifts have been noted, with fewer incidents in downtown areas but potential increases on arterial roads. However, few studies have specifically examined long-term spatial patterns in Seattle or the relationship between collision density and severity outcomes in the post-pandemic era.

This project builds on these foundations by integrating temporal, spatial, and severity analyses to address gaps in understanding how COVID-19 altered collision landscapes in Seattle.

### Research question:
Did the spatial distribution and severity of motor‑vehicle collisions in Seattle change after 2020, and are high‑density collision areas more likely to experience fatal or injurious outcomes?

### Hypothesis:
Post‑2020 collisions are less frequent and less severe, but the locations with the greatest crash density continue to account for a disproportionate share of fatalities and injuries.

### Data Source

The collision dataset used in this project was obtained from the **Seattle Open Data Portal**. The City of Seattle provides freely accessible, regularly updated records of traffic collisions, including detailed spatial coordinates and severity information. All analyses in this notebook are based on the "SDOT Collision All Years" dataset unless otherwise noted, and credit for data collection goes to the Seattle Department of Transportation through the open data initiative.

### Import

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import datetime as dt

In [ ]:
df = gpd.read_file('data/SDOT_Collision_All_Years.geojson')

In [ ]:
df.columns

In [ ]:
df.describe()

### Data Cleaning

In [ ]:
print(df.isna().sum())
print(df.isna().sum().sum())

In [ ]:
df['INCDATE'] = pd.to_datetime(df['INCDATE'])
df['YEAR'] = df['INCDATE'].dt.year
df = df[(df["YEAR"] >= 2015) & (df['YEAR'] <= 2025)]

In [ ]:
print(df.isna().sum())
print(df.isna().sum().sum())

In [ ]:
cols_to_drop = [
    'EXCEPTRSNCODE',    
    'EXCEPTRSNDESC',    
    'INATTENTIONIND',   
    'PEDROWNOTGRNT',  
    'SPEEDING',          
    'SHAREDMICROMOBILITYCD',  
    'SHAREDMICROMOBILITYDESC',
    'SDOTCOLNUM',    
    'SPDCASENO',  
    'ST_COLCODE', 
    'ST_COLDESC' 
]

df.drop(columns=cols_to_drop, inplace=True)

In [ ]:
moderate_nan_cols = ['UNDERINFL', 'WEATHER', 'ROADCOND', 'LIGHTCOND', 
                     'COLLISIONTYPE', 'JUNCTIONTYPE']

for col in moderate_nan_cols:
    df[col].fillna('Unknown', inplace=True)

### EDA

In [ ]:
df_count_year = df.groupby('YEAR').count().get(['OBJECTID']).reset_index().rename(columns={'OBJECTID': 'COUNT'})
sns.lineplot(data=df_count_year, x='YEAR', y='COUNT')
plt.title('Seattle Collisions by Year (2015-2025)')
plt.xticks(np.arange(2015, 2026, 1))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df['Day_of_Week'] = df['INCDATE'].dt.day_name()

avg_by_day = df.groupby('Day_of_Week').size() / df['Day_of_Week'].nunique()
avg_by_day

In [ ]:
sns.barplot(x=avg_by_day.index, y=avg_by_day.values, order=day_order)
plt.title('Average Number of Crashes Per Year by Day of Week')
plt.xlabel('Day of the Week')
plt.ylabel('Average Number of crashes')
plt.xticks(rotation=45)
plt.show()

In [ ]:
def get_season(month):
    if ((month > 0) & (month < 3)) | (month == 12):
        return 'Winter'
    elif (month > 2) & (month < 6):
        return 'Spring'
    elif (month > 5) & (month < 9):
        return 'Summer'
    elif (month > 8) & (month < 12):
        return 'Fall'

df = df.assign(Season=df.get('INCDATE').dt.month.apply(get_season))
df

In [ ]:
df_by_seasons = df.groupby('Season').size()
df_by_seasons

In [ ]:
sns.barplot(x= df_by_seasons.index, y=df_by_seasons.values, order=['Fall', 'Winter', 'Spring', 'Summer'])
plt.title('Total Collisions Per Year by Season')
plt.xlabel('Season')
plt.ylabel('Total Crashes')
plt.show()

In [ ]:
df_by_season_2024 = df[df.get('INCDATE').dt.year == 2024].groupby('Season').size()
df_by_season_2024

In [ ]:
sns.barplot(x= df_by_season_2024.index, y=df_by_season_2024.values, order=['Fall', 'Winter', 'Spring', 'Summer'])
plt.title('Total Collisions Per Year by Season')
plt.xlabel('Season')
plt.ylabel('Total Crashes')
plt.show()

### Analyzing vehicle dataset

In [ ]:
df_vehicle = pd.read_csv('data/SDOT_Vehicle.csv')
df_vehicle

In [ ]:
df_vehicle_type = df_vehicle.groupby('ST_VEH_TYPE_DESC').size()
df_vehicle_type

In [ ]:
# Combining vehicle type into broader categories
passenger_vehicle = ['Passenger Car', 'Taxi']
truck_suv = ['Pickup, Panel Truck or Vannette Under 10,000 lbs']
two_wheeled_vehicle = ['Motorcycle', 'Moped', 'Scooter Bike']
commercial_trucks = ['Truck (Flatbed, Van, etc)', 'Truck - Double trailer Combinations', 'Truck Tractor', 'Truck Tractor and Semi-Trailer', 'Truck and Trailer']
buses = ['Bus or Motor Stage', 'School Bus']

df_vehicle['ST_VEH_TYPE_DESC'] = df_vehicle['ST_VEH_TYPE_DESC'].replace(passenger_vehicle, 'Passenger Vehicle')
df_vehicle['ST_VEH_TYPE_DESC'] = df_vehicle['ST_VEH_TYPE_DESC'].replace(truck_suv, 'Truck/SUV')
df_vehicle['ST_VEH_TYPE_DESC'] = df_vehicle['ST_VEH_TYPE_DESC'].replace(two_wheeled_vehicle, 'Two-Wheeled')
df_vehicle['ST_VEH_TYPE_DESC'] = df_vehicle['ST_VEH_TYPE_DESC'].replace(commercial_trucks, 'Commercial Trucks')
df_vehicle['ST_VEH_TYPE_DESC'] = df_vehicle['ST_VEH_TYPE_DESC'].replace(buses, 'Buses')

df_vehicle.groupby('ST_VEH_TYPE_DESC').size()

### Merging datasets to utilize injuries and vehicle columns

In [ ]:
df_merged = pd.merge(df, df_vehicle[['COLDETKEY', 'ST_VEH_TYPE_DESC']], on='COLDETKEY', how='left')
df_merged

In [ ]:
# Combining injury severity
top_vehicle_injuries = df_merged.groupby('ST_VEH_TYPE_DESC')['INJURIES'].sum().sort_values(ascending=False)
print(f'{top_vehicle_injuries} + \n')

top_vehicle_serious_injuries = df_merged.groupby('ST_VEH_TYPE_DESC')['SERIOUSINJURIES'].sum().sort_values(ascending=False)
print(f'{top_vehicle_serious_injuries} + \n')

top_vehicle_fatalities = df_merged.groupby('ST_VEH_TYPE_DESC')['FATALITIES'].sum().sort_values(ascending=False)
print(f'{top_vehicle_fatalities} + \n')

In [ ]:
combined_data = pd.DataFrame({'Injuries': top_vehicle_injuries, 
                              'Serious_Injuries': top_vehicle_serious_injuries,
                              'Deaths': top_vehicle_fatalities})
combined_data

In [ ]:

bar_width = 0.25
indices   = np.arange(len(combined_data))

fig, ax = plt.subplots(figsize=(14, 7))
fig.subplots_adjust(top=0.85)

colors = {
    'Injuries':         ('#4fc3f7', '#0288d1'),
    'Serious_Injuries': ('#ffb74d', '#e65100'), 
    'Deaths':           ('#ef5350', '#b71c1c'), 
}

def gradient_bar(ax, x, height, width, top_color, bot_color, **kwargs):
    ax.bar(x, height * 0.85, width, color=bot_color, **kwargs)
    ax.bar(x, height,        width, color=top_color, alpha=0.55, **kwargs)

for i, (col, (tc, bc)) in enumerate(colors.items()):
    offset = (i - 1) * bar_width
    label  = col.replace('_', ' ')
    gradient_bar(ax, indices + offset, combined_data[col], bar_width, tc, bc, label=label)

for i, (col, (tc, _)) in enumerate(colors.items()):
    offset = (i - 1) * bar_width
    for x, val in zip(indices + offset, combined_data[col]):
        if val > 0:
            ax.text(x, val + combined_data.values.max() * 0.01,
                    f'{val:,}', ha='center', va='bottom',
                    fontsize=7.5, color=tc, fontweight='bold')

ax.set_xticks(indices)
ax.set_xticklabels(combined_data.index, rotation=35, ha='right', fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{int(v):,}'))

ax.set_title('Vehicle Types — Injuries & Deaths',
             fontsize=17, fontweight='bold', color='#111827', pad=20, loc='left')

patches = [mpatches.Patch(color=tc, label=col.replace('_', ' '))
           for col, (tc, _) in colors.items()]

ax.legend(handles=patches, loc='upper right', bbox_to_anchor=(1, 1),
          frameon=False, labelcolor='#111827', fontsize=11)

fig.subplots_adjust(top=0.82)  

plt.tight_layout()
plt.show()

### Spatial Data Analysis

In [ ]:
# Checking for consistency of year.
df['INCDATE'] = pd.to_datetime(df['INCDATE'])
df['YEAR'] = df.get('INCDATE').dt.year
df.groupby('YEAR').size()

In [ ]:
# Check the coordinate reference system (CRS)
print("CRS:", df.crs)
print(df.describe())

In [ ]:
# Plot recent collisions
fig, ax = plt.subplots(1, 1, figsize=(12, 10))
df.plot(ax=ax, color='blue', markersize=1, alpha=0.5)
ax.set_title('Collision Locations (2015-2025)', fontsize=16)
ax.set_xlabel('Longitude')
plt.xticks(rotation=45)
ax.set_ylabel('Latitude')
plt.show()

# Plot collisions colored by number of injuries
fig, ax = plt.subplots(1, 1, figsize=(12, 10))
inj_plot = df.plot(column='INJURIES', ax=ax, legend=True, markersize=2, alpha=0.7, cmap='Oranges', scheme='quantiles')
ax.set_title('Collision Locations Colored by Injuries (Quantiles)', fontsize=16)
ax.set_xlabel('Longitude')
plt.xticks(rotation=45)
ax.set_ylabel('Latitude')
sns.move_legend(
    inj_plot,
    'upper left',
    bbox_to_anchor=(1,1)
)
plt.show()

# Plot collisions colored by number of fatalities
fig, ax = plt.subplots(1, 1, figsize=(12, 10))
fat_plot = df.plot(column='FATALITIES', ax=ax, legend=True, markersize=2, alpha=0.7, cmap='Reds', scheme='quantiles')
ax.set_title('Collision Locations Colored by Fatalities (Quantiles)', fontsize=16)
ax.set_xlabel('Longitude')
plt.xticks(rotation=45)
ax.set_ylabel('Latitude')
sns.move_legend(
    fat_plot,
    'upper left',
    bbox_to_anchor=(1,1)
)
plt.show()

# Plot collisions colored by vehicle count
fig, ax = plt.subplots(1, 1, figsize=(12, 10))
veh_plot = df.plot(column='VEHCOUNT', ax=ax, legend=True, markersize=2, alpha=0.7, cmap='Blues', scheme='quantiles')
ax.set_title('Collision Locations Colored by Vehicle Count (Quantiles)', fontsize=16)
ax.set_xlabel('Longitude')
plt.xticks(rotation=45)
ax.set_ylabel('Latitude')
sns.move_legend(
    veh_plot,
    'upper left',
    bbox_to_anchor=(1,1)
)
plt.show()

# Plot collisions colored by total pedestrian count
df['TOTAL_PED'] = df['PEDCOUNT'] + df['PEDCYLCOUNT']
fig, ax = plt.subplots(1, 1, figsize=(12, 10))
ped_plot = df.plot(column='TOTAL_PED', ax=ax, legend=True, markersize=2, alpha=0.7, cmap='Greys', scheme='quantiles')
ax.set_title('Collision Locations Colored by Total Pedestrian Count (Quantiles)', fontsize=16)
ax.set_xlabel('Longitude')
plt.xticks(rotation=45)
ax.set_ylabel('Latitude')
sns.move_legend(
    ped_plot,
    'upper left',
    bbox_to_anchor=(1,1)
)
plt.show()

### Test

In [ ]:
from scipy import stats
from scipy.stats import gaussian_kde

In [ ]:
df = df.dropna(subset=['geometry'])
df['SEVERITY'] = df['INJURIES'] + (df['SERIOUSINJURIES'] * 3) + (df['FATALITIES'] * 5)
df_pre = df[df['YEAR'] < 2020]
df_post = df[df['YEAR'] >= 2020]

pre_2020 = df_pre['SEVERITY']
post_2020 = df_post['SEVERITY']

print(f'{pre_2020.describe()}\n')
print(f'{post_2020.describe()}\n')

In [ ]:
stat, p = stats.mannwhitneyu(pre_2020, post_2020, alternative='two-sided')
print(f'Mann-Whitney U: stat={stat:.2f}, p={p:.20f}')
print(f'Pre-2020 mean severity per collision: {pre_2020.mean():.2f}')
print(f'Post-2020 mean severity per collision: {post_2020.mean():.2f}')

In [ ]:
df['lat_bin'] = pd.cut(df.geometry.y, bins=50)
df['lon_bin'] = pd.cut(df.geometry.x, bins=50)

grid = df.groupby(['lat_bin', 'lon_bin']).agg(
    collision_count=('OBJECTID', 'count'),
    total_severity=('SEVERITY', 'sum')
).reset_index().dropna()

grid_sorted = grid.sort_values('collision_count')
grid_sorted['cum_collisions'] = grid_sorted['collision_count'].cumsum() / grid_sorted['collision_count'].sum()
grid_sorted['cum_severity'] = grid_sorted['total_severity'].cumsum() / grid_sorted['total_severity'].sum()

In [ ]:
coords_all = np.vstack([df.geometry.x, df.geometry.y])

df_severe = df[(df['FATALITIES'] > 0) | (df['SERIOUSINJURIES'] > 0)]
coords_severe = np.vstack([df_severe.geometry.x, df_severe.geometry.y])

kde_all = gaussian_kde(coords_all)
kde_severe = gaussian_kde(coords_severe)

x = np.linspace(df.geometry.x.min(), df.geometry.x.max(), 200)
y = np.linspace(df.geometry.y.min(), df.geometry.y.max(), 200)
xx, yy = np.meshgrid(x, y)
grid_points = np.vstack([xx.ravel(), yy.ravel()])

z_all = kde_all(grid_points).reshape(xx.shape)
z_severe = kde_severe(grid_points).reshape(xx.shape)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 7))
ax[0].contourf(xx, yy, z_all, cmap='YlOrRd', levels=20)
ax[0].set_title('KDE: All Collisions')
ax[1].contourf(xx, yy, z_severe, cmap='YlOrRd', levels=20)
ax[1].set_title('KDE: Severe Collisions Only')
plt.tight_layout()
plt.show()

In [ ]:
kde_pre = gaussian_kde(np.vstack([df_pre.geometry.x, df_pre.geometry.y]))
kde_post = gaussian_kde(np.vstack([df_post.geometry.x, df_post.geometry.y]))

z_pre = kde_pre(grid_points).reshape(xx.shape)
z_post = kde_post(grid_points).reshape(xx.shape)

z_diff = z_post - z_pre

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
c = ax.contourf(xx, yy, z_diff, cmap='RdBu_r', levels=20)
plt.colorbar(c, ax=ax, label='Density change (post - pre 2020)')
ax.set_title('Spatial Shift in Collision Density from Pre-2020 to Post-2020')
plt.show()

In [ ]:
downtown_mask = (
    (df.geometry.y >= 47.595) & (df.geometry.y <= 47.620) &
    (df.geometry.x >= -122.345) & (df.geometry.x <= -122.320)
    )

df_downtown = df[downtown_mask]
df_outer = df[~downtown_mask]

In [ ]:
# Plot severity over time side by side
dt_sev_year = df_downtown.groupby('YEAR')['SEVERITY'].mean().reset_index()
outer_sev_year = df_outer.groupby('YEAR')['SEVERITY'].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (label, data) in zip(axes, [('Downtown', dt_sev_year), 
                                      ('Outer Areas', outer_sev_year)]):
    ax.plot(data['YEAR'], data['SEVERITY'], marker='o', color='steelblue')
    ax.set_title(f'{label} Collision Severity Mean Over Time', fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('Collisions per unit area')
    ax.legend()

plt.suptitle('Collision Severity Mean: Downtown vs Outer Areas', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
df_downtown_pre = df_downtown[df_downtown['YEAR'] < 2020]['SEVERITY']
df_downtown_post = df_downtown[df_downtown['YEAR'] >= 2020]['SEVERITY']

print(f"Downtown pre-2020 mean severity:  {df_downtown_pre.mean():.4f}")
print(f"Downtown post-2020 mean severity: {df_downtown_post.mean():.4f}")
print(f"Change: {df_downtown_post.mean() - df_downtown_pre.mean():+.4f}")

# Same for outer
df_outer_pre = df_outer[df_outer['YEAR'] < 2020]['SEVERITY']
df_outer_post = df_outer[df_outer['YEAR'] >= 2020]['SEVERITY']

print(f"\nOuter pre-2020 mean severity:  {df_outer_pre.mean():.4f}")
print(f"Outer post-2020 mean severity: {df_outer_post.mean():.4f}")
print(f"Change: {df_outer_post.mean() - df_outer_pre.mean():+.4f}")

In [ ]:
# Count collisions per zone per year
downtown_by_year = df_downtown.groupby('YEAR').size().reset_index(name='COUNT')
outer_by_year = df_outer.groupby('YEAR').size().reset_index(name='COUNT')

downtown_area = (47.620 - 47.595) * (122.345 - 122.320)
outer_area = (df.geometry.y.max() - df.geometry.y.min()) * \
             (df.geometry.x.max() - df.geometry.x.min()) - downtown_area

downtown_by_year['DENSITY'] = downtown_by_year['COUNT'] / downtown_area
outer_by_year['DENSITY'] = outer_by_year['COUNT'] / outer_area

In [ ]:
# Plot side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (label, data) in zip(axes, [('Downtown', downtown_by_year), 
                                      ('Outer Areas', outer_by_year)]):
    ax.plot(data['YEAR'], data['DENSITY'], marker='o', color='steelblue')
    ax.set_title(f'{label} Collision Density Over Time', fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('Collisions per unit area')
    ax.legend()

plt.suptitle('Collision Density: Downtown vs Outer Areas', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Summarize the pre/post change
print("Collision Density Change Post-2020:")
for label, data in [('Downtown', downtown_by_year), ('Outer', outer_by_year)]:
    pre = data[data['YEAR'] < 2020]['DENSITY'].mean()
    post = data[data['YEAR'] >= 2020]['DENSITY'].mean()
    pct_change = ((post - pre) / pre) * 100
    print(f"{label}:")
    print(f"  Pre-2020 avg density:  {pre:.2f}")
    print(f"  Post-2020 avg density: {post:.2f}")
    print(f"  % Change: {pct_change:+.1f}%")